# 08-3 Spatial Statistics: Where Does Dengue *Really* Cluster in Taiwan?

The previous two notebooks taught you how to **draw** spatial maps (heatmaps, spot maps, choropleths). But when a patch of the map turns red, how do you know it's a **real cluster** and not your eyes inventing one? This notebook uses **spatial statistics** to turn "a feeling" into "evidence," answering three questions:

1. Is there really spatial clustering of dengue across Taiwan? → **Global Moran's I**
2. Which counties are the **cluster core / safe zone / spatial outliers**? → **Local LISA**
3. Where are the statistically significant **hot spots**? → **Getis-Ord Gi\***

> 🦟 **New stage: why switch to dengue?**
> Legionnaires' disease is a "single-building" story — one building isn't suited to county-level spatial statistics. So here we switch to **dengue × all Taiwan counties** — the most classic application of spatial epidemiology in Taiwan (the hot spots land in the south year after year). The methods you learn apply exactly the same way at a **smaller scale**: use them to find Legionella hot zones across a city's buildings and blocks.
>
> ⚠️ The case counts in this notebook are **synthetic teaching data** (designed to match Taiwan's real pattern of high-south, low-north), not actual surveillance numbers.

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
# --- Packages and font setup ---
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from libpysal.weights import Queen, KNN          # spatial weights (who is whose neighbor)
from esda.moran import Moran, Moran_Local         # global + local spatial autocorrelation
from esda.getisord import G_Local                 # Getis-Ord Gi* hot-spot analysis
from esda.smoothing import Empirical_Bayes        # disease mapping: EB smoothing

from epi_learning.viz import configure_chinese_font
configure_chinese_font()   # harmless here; keeps parity with the zh notebook

## Step 1: Load the Taiwan map + synthesize dengue data

We use the book's bundled **real Taiwan county boundaries** GeoJSON, and synthesize a **dengue incidence rate (per 100,000)** per county following the real "high in the south, low in the north and east" pattern. We also record each county's **population** and **case count** — the "small-population instability" trap later will need them.

In [ ]:
# Read the real Taiwan county boundaries (22 counties, incl. outlying islands)
gdf = gpd.read_file("data/geojson/county_smooth_inset.geojson")[
    ["COUNTYNAME", "COUNTYENG", "is_inset", "geometry"]
]

# Approximate population per county (dict keys are the Chinese names that join to COUNTYNAME)
pop = {"臺北市":2500000,"新北市":4000000,"桃園市":2270000,"臺中市":2820000,"臺南市":1870000,
       "高雄市":2750000,"基隆市":365000,"新竹市":450000,"嘉義市":265000,"新竹縣":570000,
       "苗栗縣":540000,"彰化縣":1250000,"南投縣":480000,"雲林縣":670000,"嘉義縣":500000,
       "屏東縣":810000,"宜蘭縣":454000,"花蓮縣":320000,"臺東縣":215000,
       "澎湖縣":105000,"金門縣":140000,"連江縣":13000}
# Synthetic dengue cases: south (Tainan/Kaohsiung/Pingtung) surges, north/east sparse; Lienchiang tiny pop
cases = {"臺南市":2100,"高雄市":2600,"屏東縣":710,"嘉義縣":290,"嘉義市":140,"雲林縣":270,
         "彰化縣":275,"臺中市":505,"南投縣":58,"苗栗縣":54,"桃園市":205,"新竹縣":46,"新竹市":32,
         "臺北市":150,"新北市":240,"基隆市":18,"宜蘭縣":18,"花蓮縣":10,"臺東縣":6,
         "澎湖縣":9,"金門縣":5,"連江縣":7}

gdf["population"] = gdf["COUNTYNAME"].map(pop)
gdf["cases"]      = gdf["COUNTYNAME"].map(cases)
gdf["rate"]       = (gdf["cases"] / gdf["population"] * 100000).round(1)   # incidence per 100k

# First, "eyeball it": plot the raw incidence-rate map
fig, ax = plt.subplots(figsize=(6, 7))
gdf.plot(column="rate", cmap="Reds", legend=True, edgecolor="white", linewidth=0.4, ax=ax,
         legend_kwds={"label": "Dengue rate (per 100,000)"})
ax.set_title("Raw rate map: the south looks red... but is that a REAL cluster?")
ax.axis("off")
plt.tight_layout()
plt.show()

## Why can't you judge a map "by eye"?

> 🌌 **The constellation trap**: the human brain is a "pattern-finding machine" — it strings random stars into Orion. A colored map is the same: you will **always** "see" clusters, but they may be just a random arrangement of colors.

**Spatial statistics** is a ruler that measures "is this cluster real, or did my brain invent it?" Behind it is an iron law of geography — **Tobler's First Law: "near things are more alike."** So we don't ask "is there a cluster," we ask:

> **"Is this level of similarity greater than what randomness would produce anyway?"**

The test is refreshingly blunt: **cut out** each county's number, **shuffle** them, and **randomly re-stick** them on the map many times, then check whether the real map is more concentrated than the shuffles. That's exactly what Moran's I does next.

## Step 2: First define "neighbors" — spatial weights

Before you can say "similar to its neighbors," you must define **who is a neighbor** in black and white.

> 🎲 **Monopoly-board analogy**: **Queen contiguity** — two counties are neighbors if their borders "touch" (even at a single corner), like adjacent squares on a board. The weight matrix $W$ is a roll-call of "who neighbors whom" (0/1). `transform="r"` (row-standardized) means each county's neighbor weights sum to 1 — "a fair vote among neighbors; the more neighbors, the lighter each vote."

In [ ]:
# Queen contiguity weights (borders touch = neighbor)
w_all = Queen.from_dataframe(gdf, use_index=False)

# ⚠️ The island problem: Kinmen, Penghu, and Lienchiang are islands in the sea —
# under "contiguity" they have zero neighbors!
islands = [gdf.iloc[i]["COUNTYENG"] for i in w_all.islands]
print("Counties with NO neighbors under contiguity:", islands)

See that? **Kinmen, Penghu, and Lienchiang** find no neighbors under "contiguity" — these are **islands** (disconnected units), and spatial stats can't run on them. It also demonstrates spatial analysis's number-one quirk: **change the neighbor definition and the answer changes** (more in the "traps" section).

Two common fixes:

- **KNN (k nearest neighbors)**: ignore contiguity and just grab the "k geographically nearest" units — every county is guaranteed neighbors.
- **Focus on the connected region**: the outlying islands are separated by sea with different transmission dynamics anyway, so the **cluster analysis below focuses on the 19 connected main-island counties**; those few "tiny-population, unstable-rate" islands are handled later in Step 6's "smoothing."

In [ ]:
# Focus on the 19 connected main-island counties (is_inset=False), Queen contiguity
main = gdf[~gdf["is_inset"]].reset_index(drop=True)
w = Queen.from_dataframe(main, use_index=False)
w.transform = "r"   # row-standardized
print(f"Main-island counties: {len(main)}, avg neighbors: {w.mean_neighbors:.1f}, islands: {len(w.islands)}")

# (Alternative: KNN gives ALL 22 counties neighbors, islands included)
w_knn = KNN.from_dataframe(gdf, k=4)
print(f"With KNN(k=4): all {len(gdf)} counties have neighbors, islands: {len(w_knn.islands)}")

## Step 3: Global Moran's I — the whole map's "birds-of-a-feather" index

**Global Moran's I** summarizes the whole island's spatial structure in **one number**:

- **I ≈ +1**: high counties sit next to high, low next to low (south all red, north all pale) — clean zoning.
- **I ≈ 0**: colors like scattered pepper — high and low randomly mixed, no geographic pattern.
- **I ≈ −1**: like a checkerboard, high always next to low (rare).

An I value alone isn't enough — you must ask: **"if we shuffle the numbers 999 times, would we get an I this high anyway?"** The **p-value** from shuffling tells you whether the cluster is real.

In [ ]:
y = main["rate"].values

moran = Moran(y, w, permutations=999)
print(f"Global Moran's I = {moran.I:.3f}")
print(f"p-value (999 shuffles) = {moran.p_sim:.4f}")
print(f"z-score = {moran.z_sim:.2f}")

verdict = "significant spatial clustering ✅" if moran.p_sim < 0.05 else "no detectable clustering"
print(f"\nRead: I = {moran.I:.2f} and p < 0.05 → that red patch in the south is {verdict} — not an illusion.")

## Step 4: Local LISA — where exactly is the cluster core? (the key step)

Global Moran gives the whole island **one** score; **LISA (Local Indicators of Spatial Association)** zooms in and asks **each** county: "You and your neighbors — what kind of relationship is it?" It looks at two things at once — **your own value** vs. **your neighbors' average** — sorting counties into four kinds of neighborhood:

![The four neighborhoods of LISA](../images/lisa_quadrants_en.svg)

- **HH High-High (outbreak epicentre)**: you're high, neighbors high too. You stand in the middle of the fire. → core outbreak zone, whole-area response, find the common source.
- **LL Low-Low (safe zone)**: you're low, neighbors low. → low priority, use as a control area.
- **HL High-Low (lone fire)**: you're high, neighbors all low (a spatial outlier). → **be most alert!** Could be a new independent introduction or a data anomaly.
- **LH Low-High (eye of the storm)**: you're low, neighbors all high (a spatial outlier). → maybe "not yet reached," defend fast.

> ⚠️ **esda's quadrant encoding**: in `.q`, **1=HH, 2=LH, 3=LL, 4=HL** (LH is 2, not 3!). Always cross-check before labeling.

In [ ]:
lisa = Moran_Local(y, w, permutations=999, seed=8)

# .q: 1=HH, 2=LH, 3=LL, 4=HL; keep only significant (p < 0.05) counties
labels = {1: "HH High-High (epicentre)", 2: "LH Low-High (storm eye)",
          3: "LL Low-Low (safe zone)", 4: "HL High-Low (lone fire)"}
main["lisa"] = ["not sig." if p >= 0.05 else labels[q]
                for q, p in zip(lisa.q, lisa.p_sim)]

print("Significant spatial clusters / outliers:")
for name, lab, r in zip(main["COUNTYENG"], main["lisa"], main["rate"]):
    if lab != "not sig.":
        print(f"   {name}: {lab} (rate {r})")

# Plot the LISA cluster map (left: raw rate; right: LISA categories)
color = {"HH High-High (epicentre)":"#D94452", "LL Low-Low (safe zone)":"#6A9BCC",
         "HL High-Low (lone fire)":"#D97757", "LH Low-High (storm eye)":"#9FC0E0", "not sig.":"#EDEDED"}
fig, axes = plt.subplots(1, 2, figsize=(12, 7))
main.plot(column="rate", cmap="Reds", legend=True, edgecolor="white", linewidth=0.4, ax=axes[0])
axes[0].set_title("Raw incidence rate"); axes[0].axis("off")
for lab, c in color.items():
    sub = main[main["lisa"] == lab]
    if len(sub):
        sub.plot(ax=axes[1], color=c, edgecolor="white", linewidth=0.4, label=lab)
axes[1].set_title("LISA cluster map: red=epicentre  blue=safe zone  grey=not sig.")
axes[1].axis("off"); axes[1].legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.show()

## Step 5: Hot-spot analysis Getis-Ord Gi\* — the map for decision-makers

> 🌡️ **Infrared thermal camera analogy**: Gi\* doesn't care "are you similar to your neighbors." It asks one thing — circle you together with your neighbors, and is that **circle's total heat** wildly hotter than the national average? The output is a **z-score** = "how many standard deviations hot." z ≈ +3 → glowing red (99.9% not chance); z ≈ −3 → frozen cold (significantly low); z ≈ 0 → normal temperature.

Unlike LISA, Gi\* has **no "out-of-tune" outlier category** — it just gives you a thermometer, a continuous red-to-blue spectrum — perfect for a "where to send people first" hot-spot map.

In [ ]:
# Gi* conventionally uses binary weights (star=True includes the unit itself in its circle)
w_b = Queen.from_dataframe(main, use_index=False)
w_b.transform = "B"
gi = G_Local(y, w_b, permutations=999, seed=8, star=True)

# For small samples, judge significance by the shuffle p-value, then split hot/cold by the sign of z
main["gi_z"] = gi.Zs
hot  = main.loc[(gi.p_sim < 0.05) & (gi.Zs > 0), "COUNTYENG"].tolist()
cold = main.loc[(gi.p_sim < 0.05) & (gi.Zs < 0), "COUNTYENG"].tolist()
print("Significant hot spots:", hot)
print("Significant cold spots:", cold)

fig, ax = plt.subplots(figsize=(6, 7))
main.plot(column="gi_z", cmap="RdBu_r", legend=True, edgecolor="white", linewidth=0.4, ax=ax,
          vmin=-3, vmax=3, legend_kwds={"label": "Gi* z-score (red=hot  blue=cold)"})
ax.set_title("Getis-Ord Gi* hot-spot map: the south is a statistically significant dengue hot zone")
ax.axis("off")
plt.tight_layout()
plt.show()

## Step 6: Going further — scan statistics and disease mapping (smoothing)

The three tools above (Moran's I / LISA / Gi\*) are the most common at the county level. Two more advanced tools are worth knowing conceptually (full implementations usually use R or dedicated software):

**① Kulldorff scan statistic (SaTScan)**
> 📡 **Radar-circle analogy**: sweep and expand a circle across the map to automatically find a suspicious cluster where "cases inside are unusually many, outside is normal." It catches **irregular, unknown-location** clusters and can do **space-time** scanning. Common in CDC surveillance for early warning. Tools: **SaTScan** (free software), `rsatscan`.

**② Bayesian disease mapping / spatial smoothing**
> 📷 **Sharpening a blurry photo**: small-population counties have **wildly swinging** raw rates (tiny denominators). Smoothing "borrows information from neighbors" to estimate a more stable risk. Tools: **R-INLA**, `CARBayes` (BYM models); Python has `esda.smoothing`.

**Why smoothing is needed — first see how scary "small-population instability" is:**

In [ ]:
# Lienchiang County has only 13,000 people -- its rate jumps wildly with "one more case"
p_lienchiang = 13000
print("How unstable Lienchiang County's raw rate is (population only 13,000):")
for c in [6, 7, 8]:
    print(f"   {c} cases -> rate {c / p_lienchiang * 100000:.1f} per 100k")
print("   -> just 1 more case swings the rate by ~7.7! Small-population raw rates are very unreliable.\n")

# Empirical Bayes smoothing: borrow from a data-driven prior to stabilize small-area rates
eb = Empirical_Bayes(gdf["cases"].values, gdf["population"].values)
gdf["rate_eb"] = (eb.r * 100000).round(1)
show = ["連江縣", "金門縣", "澎湖縣", "臺南市", "高雄市"]   # keys join to COUNTYNAME
print("Raw rate vs EB-smoothed (smaller population -> more correction):")
for n in show:
    r = gdf.loc[gdf["COUNTYNAME"] == n].iloc[0]
    print(f"   {r['COUNTYENG']}: raw {r['rate']:>6} -> EB {r['rate_eb']:>6} ({r['cases']} cases / {r['population']:,})")

## ⚠️ Five traps in spatial analysis

1. **MAUP (Modifiable Areal Unit Problem)**: change the spatial unit (county → township → village) and Moran's I and the hot spots can **flip entirely**. The southern cluster seen at county level might fragment or sharpen at village level. **Your conclusion is always tied to the unit you chose.**
2. **Weight-definition sensitivity**: Queen vs KNN, k=4 vs k=8 shift the results (we saw the islands change the answer in Step 2). Always re-check with a different weight definition.
3. **Multiple comparisons**: 19 counties = 19 tests, so at α=0.05 about 1 false positive shows up by luck alone. Be skeptical of "only a single county happens to be significant" (esda's `p_sim` does **not** apply FDR correction by default).
4. **Small-population instability**: small-denominator areas have very unstable raw rates (shown in Step 6). Always consider **smoothing** for small areas — don't map raw rates directly.
5. **Ecological fallacy + clustering ≠ causation**: "southern counties have high rates" ≠ "everyone in the south is at high risk," and definitely ≠ "living in the south **causes** dengue." Spatial stats only tell you **where to look**, never **why** — a significant Moran's I won't point to the mosquitoes or standing-water containers.

## Wrap-up: once you know "where," then what?

You just turned a "looks-red" map into a conclusion backed by **statistical evidence**:

- **Global Moran's I** says: dengue in Taiwan **really is** spatially clustered (not an illusion).
- **LISA** says: **the south (Tainan / Kaohsiung / Chiayi) is the cluster core (HH)**, the north is a safe zone (LL).
- **Gi\*** says: the south is a statistically significant **hot spot** — prioritize larval-source removal and spraying there.

> 🧭 **The same methods return to this book's main thread by scaling down**: swap "counties" for "the buildings / blocks within a single city," and the same Moran's I / LISA / Gi\* will find the **Legionella hot zones**.
>
> But never forget the last line: **spatial clustering only tells you "where," never "why."** After you find the hot zone, the real answer comes from **field investigation + environmental sampling + the 2×2 tables and regression** from earlier chapters. The map points the way; the evidence cracks the case.